# CPA Dataset Download & Preprocessing Pipeline

This notebook downloads all three CPA benchmark datasets from Google Drive and applies the full preprocessing pipeline described in the [CPA README](README.md). It serves as a reference for preprocessing **raw counts** data before training a CPA model.

**Datasets covered:**
| Dataset | Description |
|---|---|
| combosciplex | Combinatorial drug perturbations (sci-Plex) |
| Norman 2019 | Combinatorial CRISPR perturbations |
| Kang PBMC | IFN-β scRNA perturbation / context transfer |

**Preprocessing steps (from README):**
0. Check metadata: perturbation, dosage, cell_type, batch in `adata.obs`
1. Filter low-count cells / genes
2. Store raw counts in `adata.layers['counts']`
3. Normalize total counts
4. Log-transform
5. Highly variable gene selection

## 0. Imports

In [2]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import scanpy as sc
import anndata as ad
import gdown

sc.settings.verbosity = 2
plt.rcParams.update({'figure.dpi': 100, 'figure.figsize': (6, 4)})

# Output directory: datasets/ relative to the notebook's working directory
DATA_DIR = os.path.join(os.getcwd(), 'datasets')
os.makedirs(DATA_DIR, exist_ok=True)
print(f'Dataset directory: {DATA_DIR}')
print(f'scanpy version: {sc.__version__}')


Dataset directory: /Users/cheng.wang/Documents/GitHub/cpa/datasets
scanpy version: 1.11.3


## 1. Download Datasets

Files are downloaded from Google Drive via `gdown`. Downloads are skipped if the file already exists.

In [3]:
DATASETS = {
    'combosciplex': {
        'gdrive_id': '1RRV0_qYKGTvD3oCklKfoZQFYqKJy4l6t',
        'filename': 'combo_sciplex_prep_hvg_filtered.h5ad',
        'description': 'Combinatorial drug perturbations (sci-Plex)',
        'perturbation_key': 'condition',
        'dosage_key': 'dosage',
        'control': 'CHEMBL504',
    },
    'norman': {
        'gdrive_id': '109G9MmL-8-uh7OSjnENeZ5vFbo62kI7j',
        'filename': 'Norman2019_normalized_hvg.h5ad',
        'description': 'Combinatorial CRISPR perturbations (Norman 2019)',
        'perturbation_key': 'condition',
        'dosage_key': 'dosage',
        'control': 'ctrl',
    },
    'kang': {
        'gdrive_id': '1z8gGKQ6oDoi2blCU2IVihKA38h5fORRp',
        'filename': 'kang_normalized_hvg.h5ad',
        'description': 'IFN-β PBMC scRNA perturbation (Kang 2018)',
        'perturbation_key': 'condition',
        'dosage_key': 'dosage',
        'control': 'ctrl',
    },
}

for name, info in DATASETS.items():
    dst = os.path.join(DATA_DIR, info['filename'])
    if os.path.exists(dst):
        print(f'[{name}] Already exists: {dst}')
    else:
        url = f"https://drive.google.com/uc?id={info['gdrive_id']}"
        print(f'[{name}] Downloading {info["filename"]} ...')
        gdown.download(url, dst, quiet=False)
        print(f'[{name}] Saved to {dst}')


[combosciplex] Already exists: /Users/cheng.wang/Documents/GitHub/cpa/datasets/combo_sciplex_prep_hvg_filtered.h5ad
[norman] Already exists: /Users/cheng.wang/Documents/GitHub/cpa/datasets/Norman2019_normalized_hvg.h5ad
[kang] Already exists: /Users/cheng.wang/Documents/GitHub/cpa/datasets/kang_normalized_hvg.h5ad


## 2. Preprocessing Pipeline

Each dataset is loaded, inspected, and run through the full preprocessing pipeline.

> **Note:** The downloaded `.h5ad` files are already preprocessed (normalized + HVG-filtered) and contain raw counts in `adata.layers['counts']`. This pipeline demonstrates the canonical steps by resetting `adata.X` to raw counts before reapplying — making it directly applicable to **any custom raw scRNA-seq dataset**.

In [4]:
def preprocess_adata(
    adata: ad.AnnData,
    name: str,
    perturbation_key: str,
    dosage_key: str,
    control: str,
    min_counts_cells: int = 100,
    min_counts_genes: int = 5,
    n_top_genes: int = 5000,
) -> ad.AnnData:
    """
    Full CPA preprocessing pipeline (README recipe).

    Steps:
      0. Metadata validation
      1. Filter cells (and optionally genes)
      2. Store raw counts in layers['counts']
      3. Normalize total
      4. Log1p transform
      5. Highly variable gene selection
    """
    print(f'\n========== {name} ==========')
    print(f'Input: {adata.n_obs} cells x {adata.n_vars} genes')

    # -------------------------------------------------------------------------
    # Step 0: Metadata validation
    # -------------------------------------------------------------------------
    print('\n[Step 0] Checking metadata ...')
    optional_obs = ['cell_type', 'batch']

    if perturbation_key not in adata.obs.columns:
        fallback = None
        for cand in ['condition', 'perturbation', 'drug', 'treatment']:
            if cand in adata.obs.columns:
                fallback = cand
                break
        if fallback is None:
            raise ValueError(
                f"Required perturbation column '{perturbation_key}' not found in adata.obs. "
                f"Available: {list(adata.obs.columns)}"
            )
        print(f"  - '{perturbation_key}' not found, using fallback '{fallback}'")
        perturbation_key = fallback

    print(f'  ✓ {perturbation_key}: {adata.obs[perturbation_key].nunique()} unique values')

    for col in optional_obs:
        if col in adata.obs.columns:
            print(f'  ✓ {col} (optional): {adata.obs[col].nunique()} unique values')
        else:
            print(f'  - {col} (optional): not found')

    # Create dosage column if missing.
    if dosage_key not in adata.obs.columns:
        print(f"  - '{dosage_key}' not found, creating dummy dosage from '{perturbation_key}'")
        adata.obs[dosage_key] = (
            adata.obs[perturbation_key]
            .astype(str)
            .apply(lambda x: '+'.join(['1.0' for _ in x.split('+')]))
            .values
        )
    print(f'  ✓ {dosage_key}: {adata.obs[dosage_key].nunique()} unique values')

    # -------------------------------------------------------------------------
    # Reset X to raw counts (if available in layers)
    # -------------------------------------------------------------------------
    if 'counts' in adata.layers:
        print('\n[Pre-step] Resetting adata.X to raw counts from layers["counts"] ...')
        adata.X = adata.layers['counts'].copy()
    else:
        print('\n[Pre-step] No counts layer found - assuming adata.X contains raw counts.')

    # -------------------------------------------------------------------------
    # Step 1: Filter cells and genes
    # -------------------------------------------------------------------------
    print(f'\n[Step 1] Filtering cells (min_counts={min_counts_cells}) ...')
    n_before = adata.n_obs
    sc.pp.filter_cells(adata, min_counts=min_counts_cells)
    print(f'  Cells: {n_before} -> {adata.n_obs} (removed {n_before - adata.n_obs})')

    if min_counts_genes > 0:
        n_genes_before = adata.n_vars
        sc.pp.filter_genes(adata, min_counts=min_counts_genes)
        print(f'  Genes (min_counts={min_counts_genes}): {n_genes_before} -> {adata.n_vars}')

    # -------------------------------------------------------------------------
    # Step 2: Store raw counts
    # -------------------------------------------------------------------------
    print('\n[Step 2] Storing raw counts in layers["counts"] ...')
    adata.layers['counts'] = adata.X.copy()
    print('  ✓ adata.layers["counts"] saved')

    # -------------------------------------------------------------------------
    # Step 3: Normalize
    # -------------------------------------------------------------------------
    print('\n[Step 3] Normalizing total counts (target_sum=1e4) ...')
    sc.pp.normalize_total(adata, target_sum=1e4, exclude_highly_expressed=True)
    print('  ✓ Normalization done')

    # -------------------------------------------------------------------------
    # Step 4: Log transform
    # -------------------------------------------------------------------------
    print('\n[Step 4] Log1p transform ...')
    sc.pp.log1p(adata)
    print('  ✓ Log1p done')

    # -------------------------------------------------------------------------
    # Step 5: HVG selection
    # -------------------------------------------------------------------------
    print(f'\n[Step 5] Selecting top {n_top_genes} highly variable genes ...')
    batch_key = 'batch' if 'batch' in adata.obs.columns else None
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=n_top_genes,
        subset=True,
        batch_key=batch_key,
    )
    print(f'  ✓ HVG subset: {adata.n_obs} cells x {adata.n_vars} genes')

    print(f'\n[Done] Final shape: {adata.n_obs} cells x {adata.n_vars} genes')
    return adata


### 2.1 combosciplex (step-by-step debug)

In [5]:
info = DATASETS['combosciplex']
path = os.path.join(DATA_DIR, info['filename'])
adata_combo = sc.read_h5ad(path)
print('Loaded:', adata_combo)
print('obs columns:', list(adata_combo.obs.columns))
print('layers:', list(adata_combo.layers.keys()))

Loaded: AnnData object with n_obs × n_vars = 63378 × 5000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway1', 'pathway2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'tissue_type', 'organism', 'perturbation_type', 'ngenes', 'percent_mito', 'percent_ribo', 'nperts', 'chembl-ID', 'condition', 'condition_ID', 'control', 'cell_type', 'smiles_rdkit', 'source', 'sample', 'Size_Factor', 'n.umi', 'RT_well', 'Drug1', 'Drug2', 'Well', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'leiden', 'split', 'condition_old', 'pert_type', 'batch', 'split_1ct_MEC', 'split_2ct_MEC', 'split_3ct_MEC', 'batch_cov', 'batch_cov_cond', 'log_dose', 'cov_drug_dose'
    var: 'ensembl_id-0', 'ncounts-0', 'ncells-0', 'symbol-0', 'symbol-1', 'id-1', 'n_cells-1', 'mt-1', 'n_cells_by_counts-1', 'mean_counts-1', 'pct_dropout_by_counts-1', 'total_counts-1', 'highly_variable-1', 'means-1', 'dispersions-1', 'dispers

In [6]:
# Initialize working copy
adata_combo_work = adata_combo.copy()
perturbation_key = info['perturbation_key']
dosage_key = info['dosage_key']
control = info['control']

print(f"\n========== combosciplex ==========")
print(f"Input: {adata_combo_work.n_obs} cells x {adata_combo_work.n_vars} genes")
print(f"Specified perturbation_key: {perturbation_key}")
print(f"Specified dosage_key: {dosage_key}")



========== combosciplex ==========
Input: 63378 cells x 5000 genes
Specified perturbation_key: condition
Specified dosage_key: dosage


In [7]:
adata_combo.obs['Drug2'].value_counts()

Drug2
Dasatinib       7950
PCI-34051       7273
SRT2104         7080
Cediranib       5799
Curcumin        4980
Sorafenib       4747
Crizotinib      4303
SRT1720         4086
Carmofur        2692
Alvespimycin    2274
Danusertib      1939
SRT3025         1889
Dacinostat      1869
Givinostat      1682
Panobinostat    1578
control         1451
Tanespimycin    1310
Pirarubicin      476
Name: count, dtype: int64

In [8]:
# Step 0: Metadata validation
print("\n[Step 0] Checking metadata ...")
optional_obs = ['cell_type', 'batch']

if perturbation_key not in adata_combo_work.obs.columns:
    fallback = None
    for cand in ['condition', 'perturbation', 'drug', 'treatment']:
        if cand in adata_combo_work.obs.columns:
            fallback = cand
            break
    if fallback is None:
        raise ValueError(
            f"Required perturbation column '{perturbation_key}' not found in adata.obs. "
            f"Available: {list(adata_combo_work.obs.columns)}"
        )
    print(f"  - '{perturbation_key}' not found, using fallback '{fallback}'")
    perturbation_key = fallback

print(f"  ✓ {perturbation_key}: {adata_combo_work.obs[perturbation_key].nunique()} unique values")

for col in optional_obs:
    if col in adata_combo_work.obs.columns:
        print(f"  ✓ {col} (optional): {adata_combo_work.obs[col].nunique()} unique values")
    else:
        print(f"  - {col} (optional): not found")

# Create dosage column if missing
if dosage_key not in adata_combo_work.obs.columns:
    print(f"  - '{dosage_key}' not found, creating dummy dosage from '{perturbation_key}'")
    adata_combo_work.obs[dosage_key] = (
        adata_combo_work.obs[perturbation_key]
        .astype(str)
        .apply(lambda x: '+'.join(['1.0' for _ in x.split('+')]))
        .values
    )
print(f"  ✓ {dosage_key}: {adata_combo_work.obs[dosage_key].nunique()} unique values")



[Step 0] Checking metadata ...
  ✓ condition: 32 unique values
  ✓ cell_type (optional): 1 unique values
  ✓ batch (optional): 1 unique values
  - 'dosage' not found, creating dummy dosage from 'condition'
  ✓ dosage: 2 unique values


In [9]:
# Pre-step: Reset X to raw counts (if available in layers)
if 'counts' in adata_combo_work.layers:
    print("\n[Pre-step] Resetting adata.X to raw counts from layers['counts'] ...")
    adata_combo_work.X = adata_combo_work.layers['counts'].copy()
    print(f"  ✓ X shape: {adata_combo_work.X.shape}")
else:
    print("\n[Pre-step] No counts layer found - assuming adata.X contains raw counts.")



[Pre-step] Resetting adata.X to raw counts from layers['counts'] ...
  ✓ X shape: (63378, 5000)


In [10]:
# Step 1: Filter cells and genes
min_counts_cells = 100
min_counts_genes = 5

print(f"\n[Step 1] Filtering cells (min_counts={min_counts_cells}) ...")
n_before = adata_combo_work.n_obs
sc.pp.filter_cells(adata_combo_work, min_counts=min_counts_cells)
print(f"  Cells: {n_before} -> {adata_combo_work.n_obs} (removed {n_before - adata_combo_work.n_obs})")

if min_counts_genes > 0:
    n_genes_before = adata_combo_work.n_vars
    sc.pp.filter_genes(adata_combo_work, min_counts=min_counts_genes)
    print(f"  Genes (min_counts={min_counts_genes}): {n_genes_before} -> {adata_combo_work.n_vars}")



[Step 1] Filtering cells (min_counts=100) ...
  Cells: 63378 -> 63378 (removed 0)
  Genes (min_counts=5): 5000 -> 5000


In [11]:
# Step 2: Store raw counts in layers
print("\n[Step 2] Storing raw counts in layers['counts'] ...")
adata_combo_work.layers['counts'] = adata_combo_work.X.copy()
print(f"  ✓ adata.layers['counts'] saved (shape: {adata_combo_work.layers['counts'].shape})")
print(f"  Available layers: {list(adata_combo_work.layers.keys())}")



[Step 2] Storing raw counts in layers['counts'] ...
  ✓ adata.layers['counts'] saved (shape: (63378, 5000))
  Available layers: ['counts']


In [12]:
# Step 3: Normalize total counts
print("\n[Step 3] Normalizing total counts (target_sum=1e4) ...")
sc.pp.normalize_total(adata_combo_work, target_sum=1e4, exclude_highly_expressed=True)
print(f"  ✓ Normalization done")
print(f"  X stats: min={adata_combo_work.X.min():.4f}, max={adata_combo_work.X.max():.4f}, mean={adata_combo_work.X.mean():.4f}")



[Step 3] Normalizing total counts (target_sum=1e4) ...


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


  ✓ Normalization done
  X stats: min=0.0000, max=1702.1277, mean=2.1330


In [13]:
# Step 4: Log1p transform
print("\n[Step 4] Log1p transform ...")
sc.pp.log1p(adata_combo_work)
print(f"  ✓ Log1p done")
print(f"  X stats after log: min={adata_combo_work.X.min():.4f}, max={adata_combo_work.X.max():.4f}, mean={adata_combo_work.X.mean():.4f}")



[Step 4] Log1p transform ...
  ✓ Log1p done
  X stats after log: min=0.0000, max=7.4402, mean=0.2632


In [14]:
# Step 5: Highly variable gene selection
n_top_genes = 5000

print(f"\n[Step 5] Selecting top {n_top_genes} highly variable genes ...")
batch_key = 'batch' if 'batch' in adata_combo_work.obs.columns else None
if batch_key:
    print(f"  Using batch_key='{batch_key}' for HVG selection")
sc.pp.highly_variable_genes(
    adata_combo_work,
    n_top_genes=n_top_genes,
    subset=True,
    batch_key=batch_key,
)
print(f"  ✓ HVG subset: {adata_combo_work.n_obs} cells x {adata_combo_work.n_vars} genes")

# Assign to final variable
adata_combo_pp = adata_combo_work
print(f"\n[Done] Final shape: {adata_combo_pp.n_obs} cells x {adata_combo_pp.n_vars} genes")



[Step 5] Selecting top 5000 highly variable genes ...
  Using batch_key='batch' for HVG selection
  ✓ HVG subset: 63378 cells x 5000 genes

[Done] Final shape: 63378 cells x 5000 genes


### 2.2 Norman 2019 (CRISPR)

In [15]:
info = DATASETS['norman']
path = os.path.join(DATA_DIR, info['filename'])
adata_norman = sc.read_h5ad(path)
print('Loaded:', adata_norman)
print('obs columns:', list(adata_norman.obs.columns))
print('layers:', list(adata_norman.layers.keys()))

Loaded: AnnData object with n_obs × n_vars = 111122 × 5044
    obs: 'guide_id', 'read_count', 'UMI_count', 'coverage', 'gemgroup', 'good_coverage', 'number_of_cells', 'tissue_type', 'cell_line', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'perturbation', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo', 'n_counts', 'condition', 'pert_type', 'cell_type', 'source', 'condition_ID', 'control', 'dose_value', 'pathway', 'cov_cond', 'pert', 'split_hardest', 'split_1', 'split_2', 'split_3', 'split_4', 'split_5', 'split_6', 'cond_harm'
    var: 'ensemble_id', 'ncounts', 'ncells', 'symbol', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'cell_type_colors', 'gene_embedding_path', 'hvg', 'log1p', 'neighbors', 'rank_genes_groups_cov', 'source_colors', 'split_1_colors', 'split_2_colors', 'split_3_colors', 'split_4_colors', 'split_5_colors', 'split_hardest_colors', 'umap'
    obsm: 'X_pca', 'X_umap'
    layers: 'counts'
    obsp: 'connectivit

In [16]:
# Check if dosage exists; if not it will be auto-created as dummy
info = DATASETS['norman']
adata_norman_pp = preprocess_adata(
    adata_norman.copy(),
    name='Norman 2019',
    perturbation_key=info['perturbation_key'],
    dosage_key=info['dosage_key'],
    control=info['control'],
)


========== Norman 2019 ==========
Input: 111122 cells x 5044 genes

[Step 0] Checking metadata ...
  ✓ condition: 235 unique values
  ✓ cell_type (optional): 1 unique values
  - batch (optional): not found
  - 'dosage' not found, creating dummy dosage from 'condition'
  ✓ dosage: 2 unique values

[Pre-step] Resetting adata.X to raw counts from layers["counts"] ...

[Step 1] Filtering cells (min_counts=100) ...
  Cells: 111122 -> 111122 (removed 0)
  Genes (min_counts=5): 5044 -> 4215

[Step 2] Storing raw counts in layers["counts"] ...
  ✓ adata.layers["counts"] saved

[Step 3] Normalizing total counts (target_sum=1e4) ...


adata.X seems to be already log-transformed.


  ✓ Normalization done

[Step 4] Log1p transform ...
  ✓ Log1p done

[Step 5] Selecting top 5000 highly variable genes ...
  ✓ HVG subset: 111122 cells x 4215 genes

[Done] Final shape: 111122 cells x 4215 genes


### 2.3 Kang PBMC (IFN-β)

In [17]:
info = DATASETS['kang']
path = os.path.join(DATA_DIR, info['filename'])
adata_kang = sc.read_h5ad(path)
print('Loaded:', adata_kang)
print('obs columns:', list(adata_kang.obs.columns))
print('layers:', list(adata_kang.layers.keys()))

Loaded: AnnData object with n_obs × n_vars = 13576 × 5000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'stim', 'seurat_annotations', 'integrated_snn_res.0.5', 'seurat_clusters', 'condition', 'cell_type', 'cov_cond', 'split_CD14 Mono', 'split_CD4 T', 'split_T', 'split_CD8 T', 'split_B', 'split_DC', 'split_CD16 Mono', 'split_NK'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'symbol'
    uns: 'hvg', 'log1p', 'rank_genes_groups_cov'
    layers: 'counts'
obs columns: ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'stim', 'seurat_annotations', 'integrated_snn_res.0.5', 'seurat_clusters', 'condition', 'cell_type', 'cov_cond', 'split_CD14 Mono', 'split_CD4 T', 'split_T', 'split_CD8 T', 'split_B', 'split_DC', 'split_CD16 Mono', 'split_NK']
layers: ['counts']


In [18]:
info = DATASETS['kang']
adata_kang_pp = preprocess_adata(
    adata_kang.copy(),
    name='Kang PBMC',
    perturbation_key=info['perturbation_key'],
    dosage_key=info['dosage_key'],
    control=info['control'],
)


========== Kang PBMC ==========
Input: 13576 cells x 5000 genes

[Step 0] Checking metadata ...
  ✓ condition: 2 unique values
  ✓ cell_type (optional): 8 unique values
  - batch (optional): not found
  - 'dosage' not found, creating dummy dosage from 'condition'
  ✓ dosage: 1 unique values

[Pre-step] Resetting adata.X to raw counts from layers["counts"] ...

[Step 1] Filtering cells (min_counts=100) ...
  Cells: 13576 -> 13576 (removed 0)
  Genes (min_counts=5): 5000 -> 4823

[Step 2] Storing raw counts in layers["counts"] ...
  ✓ adata.layers["counts"] saved

[Step 3] Normalizing total counts (target_sum=1e4) ...


adata.X seems to be already log-transformed.


  ✓ Normalization done

[Step 4] Log1p transform ...
  ✓ Log1p done

[Step 5] Selecting top 5000 highly variable genes ...
  ✓ HVG subset: 13576 cells x 4823 genes

[Done] Final shape: 13576 cells x 4823 genes


## 3. QC Visualization

Distribution of total counts and number of genes per cell, before and after filtering.

In [19]:
datasets_pp = {
    'combosciplex': adata_combo_pp,
    'Norman 2019': adata_norman_pp,
    'Kang PBMC': adata_kang_pp,
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('QC Metrics (after preprocessing)', fontsize=14, y=1.02)

for col_idx, (dname, adata_pp) in enumerate(datasets_pp.items()):
    # Total counts distribution
    ax = axes[0, col_idx]
    counts = np.asarray(adata_pp.layers['counts'].sum(axis=1)).flatten()
    ax.hist(counts, bins=60, color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(dname)
    ax.set_xlabel('Total counts (raw)')
    ax.set_ylabel('# cells')
    ax.axvline(100, color='red', linestyle='--', linewidth=1, label='min_counts=100')
    ax.legend(fontsize=8)

    # Number of expressed genes per cell
    ax2 = axes[1, col_idx]
    n_genes = np.asarray((adata_pp.layers['counts'] > 0).sum(axis=1)).flatten()
    ax2.hist(n_genes, bins=60, color='coral', edgecolor='none', alpha=0.8)
    ax2.set_title(dname)
    ax2.set_xlabel('# expressed genes')
    ax2.set_ylabel('# cells')

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'qc_plots.png'), dpi=120, bbox_inches='tight')
plt.show()
print('QC plot saved to datasets/qc_plots.png')

QC plot saved to datasets/qc_plots.png


## 4. Perturbation Composition Plots

In [20]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Top 20 Perturbations per Dataset', fontsize=14)

for col_idx, (dname, (adata_pp, info_key)) in enumerate(zip(
    datasets_pp.keys(),
    [
        (adata_combo_pp, DATASETS['combosciplex']['perturbation_key']),
        (adata_norman_pp, DATASETS['norman']['perturbation_key']),
        (adata_kang_pp, DATASETS['kang']['perturbation_key']),
    ]
)):
    adata_pp2, pk = list(datasets_pp.values())[col_idx], list(DATASETS.values())[col_idx]['perturbation_key']
    top20 = adata_pp2.obs[pk].value_counts().head(20)
    ax = axes[col_idx]
    ax.barh(top20.index[::-1], top20.values[::-1], color='steelblue')
    ax.set_title(dname)
    ax.set_xlabel('# cells')
    ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'perturbation_composition.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Perturbation composition plot saved.')

Perturbation composition plot saved.


## 5. Save Preprocessed Datasets

In [21]:
save_map = {
    'combosciplex_preprocessed.h5ad': adata_combo_pp,
    'Norman2019_preprocessed.h5ad': adata_norman_pp,
    'kang_preprocessed.h5ad': adata_kang_pp,
}

for filename, adata_pp in save_map.items():
    out_path = os.path.join(DATA_DIR, filename)
    adata_pp.write_h5ad(out_path)
    print(f'Saved: {out_path}  ({adata_pp.n_obs} cells x {adata_pp.n_vars} genes)')

Saved: /Users/cheng.wang/Documents/GitHub/cpa/datasets/combosciplex_preprocessed.h5ad  (63378 cells x 5000 genes)
Saved: /Users/cheng.wang/Documents/GitHub/cpa/datasets/Norman2019_preprocessed.h5ad  (111122 cells x 4215 genes)
Saved: /Users/cheng.wang/Documents/GitHub/cpa/datasets/kang_preprocessed.h5ad  (13576 cells x 4823 genes)


## Summary

All three datasets have been downloaded, preprocessed, and saved to `datasets/`.

| File | Contents |
|---|---|
| `combosciplex_preprocessed.h5ad` | Preprocessed combinatorial drug data |
| `Norman2019_preprocessed.h5ad` | Preprocessed CRISPR perturbation data |
| `kang_preprocessed.h5ad` | Preprocessed IFN-β PBMC data |
| `qc_plots.png` | QC distribution plots |
| `perturbation_composition.png` | Perturbation composition plots |

These files are ready to be used as input for CPA model training. See the tutorials in `docs/tutorials/` for next steps.